# Notebook 3 — ML Models and SHAP Interpretability

Trains XGBoost, Random Forest, and SVM classifiers to predict urban heat
hotspots from landscape predictors, then interprets the models using SHAP.

**Prerequisites:**
- Processed raster stack: `data/processed/ouaga_aligned_stack.tif`
  (produced by `01_processing_pipeline.ipynb`)

**Outputs (saved to `figures/pub/`):**
- `shap_bar_importance.png` — global feature importance
- `shap_beeswarm.png` — direction of influence
- `shap_dependence_grid.png` — feature value vs SHAP value
- `susceptibility_maps.png` — XGB/RF/SVM side-by-side
- `binary_hotspot_map.png` — XGBoost binary classification

**Configuration / reproduction note:**
- `MODEL_MODE = "load"` (default) reads the frozen pickled models in
  `models/` (archived on Zenodo) and reproduces the exact published
  numbers. **Use this to verify results.**
- `MODEL_MODE = "manual"` retrains with the published best
  hyperparameters (~2 min). Deterministic given fixed `random_state=42`,
  but may differ from the pickled models due to library-version drift.
- `MODEL_MODE = "grid_search"` redoes the full CV tuning (hours).

In [ ]:
from pathlib import Path

import joblib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from matplotlib.patches import Patch
from rasterio.warp import transform_bounds
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src.data import load_dataset

# -- Config ----------------------------------------------------------------
MODEL_MODE = "load"  # "load" | "manual" | "grid_search"

PROJECT_ROOT = Path("..").resolve()
FIG_DIR = PROJECT_ROOT / "figures" / "pub"
MODEL_DIR = PROJECT_ROOT / "models"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_PARAMS_XGB = {
    "n_estimators": 500,
    "max_depth": 9,
    "learning_rate": 0.2,
    "reg_alpha": 0,
    "reg_lambda": 0.001,
}

BEST_PARAMS_RF = {
    "n_estimators": 300,
    "max_depth": None,
}

BEST_PARAMS_SVM = {
    "C": 100,
    "gamma": "scale",
    "kernel": "rbf",
}

## Load data

In [ ]:
df, config = load_dataset("../config/processing.yaml")

# Predictors = all bands except LST and hotspot (the last two)
PREDICTOR_COLS = config["band_names"][:-2]

X = df[PREDICTOR_COLS].values
y = df["hotspot"].values

print(f"Dataset: {X.shape[0]:,} valid pixels, {X.shape[1]} predictors")
print(f"Predictors: {PREDICTOR_COLS}")
print(f"Class balance: {np.sum(y == 0):,} non-hotspot ({100*np.mean(y == 0):.1f}%), "
      f"{np.sum(y == 1):,} hotspot ({100*np.mean(y == 1):.1f}%)")

## Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Testing:  {X_test.shape[0]:,} samples")

## Model training

In [ ]:
if MODEL_MODE == "grid_search":
    print("Running grid search (this will take a while)...")

    param_grid_xgb = {
        "n_estimators": [200, 300, 500],
        "max_depth": [2, 5, 7, 9],
        "learning_rate": [0.05, 0.1, 0.2],
        "reg_alpha": [0, 0.00001, 0.0001, 0.001],
        "reg_lambda": [0, 0.001, 0.01],
    }
    grid_xgb = GridSearchCV(
        xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
        param_grid_xgb, cv=5, scoring="accuracy", n_jobs=-1, verbose=1,
    )
    grid_xgb.fit(X_train, y_train)
    xgb_model = grid_xgb.best_estimator_
    print(f"XGBoost best params: {grid_xgb.best_params_}")

    param_grid_rf = {"n_estimators": [50, 100, 200, 300], "max_depth": [None, 5, 10]}
    grid_rf = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid_rf, cv=5, scoring="accuracy", n_jobs=-1, verbose=1,
    )
    grid_rf.fit(X_train, y_train)
    rf_model = grid_rf.best_estimator_
    print(f"RF best params: {grid_rf.best_params_}")

    svm_pipe = Pipeline([("scaler", StandardScaler()), ("svc", SVC(probability=True, random_state=42))])
    param_grid_svm = {
        "svc__C": [1, 10, 100],
        "svc__kernel": ["rbf"],
        "svc__gamma": [0.01, 0.05, 0.1, "scale"],
    }
    grid_svm = GridSearchCV(svm_pipe, param_grid_svm, cv=5, scoring="accuracy", n_jobs=-1)
    grid_svm.fit(X_train, y_train)
    svm_model = grid_svm.best_estimator_
    print(f"SVM best params: {grid_svm.best_params_}")

    joblib.dump(xgb_model, MODEL_DIR / "xgb_model.pkl")
    joblib.dump(rf_model, MODEL_DIR / "rf_model.pkl")
    joblib.dump(svm_model, MODEL_DIR / "svm_model.pkl")
    print("Models saved to models/")

elif MODEL_MODE == "manual":
    print("Fitting with known best parameters...")

    xgb_model = xgb.XGBClassifier(
        **BEST_PARAMS_XGB, random_state=42, eval_metric="logloss"
    )
    xgb_model.fit(X_train, y_train)

    rf_model = RandomForestClassifier(**BEST_PARAMS_RF, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    svm_model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(**BEST_PARAMS_SVM, probability=True, random_state=42)),
    ])
    svm_model.fit(X_train, y_train)

    joblib.dump(xgb_model, MODEL_DIR / "xgb_model.pkl")
    joblib.dump(rf_model, MODEL_DIR / "rf_model.pkl")
    joblib.dump(svm_model, MODEL_DIR / "svm_model.pkl")
    print("Models trained and saved to models/")

elif MODEL_MODE == "load":
    print("Loading saved models...")
    xgb_model = joblib.load(MODEL_DIR / "xgb_model.pkl")
    rf_model = joblib.load(MODEL_DIR / "rf_model.pkl")
    svm_model = joblib.load(MODEL_DIR / "svm_model.pkl")
    print("Models loaded from models/")

else:
    raise ValueError(f"Unknown MODEL_MODE: '{MODEL_MODE}'")

## Model evaluation

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    """Print and return test-set classification metrics."""
    y_pred = model.predict(X_test)
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "Kappa": cohen_kappa_score(y_test, y_pred),
    }
    print(f"\n{name}: Acc={metrics['Accuracy']:.3f}, "
          f"Prec={metrics['Precision']:.3f}, Rec={metrics['Recall']:.3f}, "
          f"F1={metrics['F1']:.3f}, Kappa={metrics['Kappa']:.3f}")
    return metrics


results = [
    evaluate_model(xgb_model, X_test, y_test, "XGBoost"),
    evaluate_model(rf_model, X_test, y_test, "Random Forest"),
    evaluate_model(svm_model, X_test, y_test, "SVM"),
]

results_df = pd.DataFrame(results)
print("\n")
print(results_df.to_string(index=False))

## Full-map prediction and GeoTIFF export

In [ ]:
import rasterio

# Predict on the full raster (including NaN areas)
data_3d = config["raster_info"]["data_3d"]
bands, rows, cols = data_3d.shape

# Build feature matrix for all pixels (predictors only)
n_predictors = len(PREDICTOR_COLS)
X_full = data_3d[:n_predictors].reshape(n_predictors, rows * cols).T

# Mask for valid pixels
valid_mask = ~np.isnan(X_full).any(axis=1)

# Predict only on valid pixels
pred_full = np.full(rows * cols, np.nan)
pred_full[valid_mask] = xgb_model.predict(X_full[valid_mask])
pred_map = pred_full.reshape(rows, cols)

# Save GeoTIFF
meta = config["raster_info"]["meta"].copy()
meta.update({"count": 1, "dtype": "float32"})

out_dir = PROJECT_ROOT / "data" / "processed"
hotspot_path = out_dir / "ouaga_predicted_hotspots.tif"
with rasterio.open(hotspot_path, "w", **meta) as dst:
    dst.write(pred_map.astype("float32"), 1)

print(f"Saved: {hotspot_path}")

## SHAP Interpretability

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Bar plot: global feature importance
fig, ax = plt.subplots(figsize=(8, 5))
shap.summary_plot(
    shap_values, X_test,
    feature_names=PREDICTOR_COLS,
    plot_type="bar",
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_bar_importance.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_bar_importance.png'}")

In [ ]:
# Beeswarm plot: direction of influence
fig, ax = plt.subplots(figsize=(8, 5))
shap.summary_plot(
    shap_values, X_test,
    feature_names=PREDICTOR_COLS,
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_beeswarm.png'}")

In [ ]:
# Dependence scatter plots (8-panel grid)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=300)

for idx, (feature, ax) in enumerate(zip(PREDICTOR_COLS, axes.flat)):
    ax.scatter(
        X_test[:, idx], shap_values[:, idx],
        alpha=0.4, s=6, c="steelblue", edgecolors="none",
    )
    ax.axhline(y=0, color="red", linestyle="--", linewidth=0.5, alpha=0.7)
    ax.set_xlabel(feature, fontsize=9)
    ax.set_ylabel("SHAP value", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.text(0.02, 0.98, f"({chr(97 + idx)})",
            transform=ax.transAxes, fontsize=10, fontweight="bold",
            verticalalignment="top")

plt.tight_layout()
plt.savefig(FIG_DIR / "shap_dependence_grid.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_dependence_grid.png'}")

## Susceptibility Maps

In [ ]:
# Probability predictions for all models
prob_xgb = np.full(rows * cols, np.nan)
prob_rf = np.full(rows * cols, np.nan)
prob_svm = np.full(rows * cols, np.nan)

prob_xgb[valid_mask] = xgb_model.predict_proba(X_full[valid_mask])[:, 1]
prob_rf[valid_mask] = rf_model.predict_proba(X_full[valid_mask])[:, 1]
prob_svm[valid_mask] = svm_model.predict_proba(X_full[valid_mask])[:, 1]

prob_xgb_map = prob_xgb.reshape(rows, cols)
prob_rf_map = prob_rf.reshape(rows, cols)
prob_svm_map = prob_svm.reshape(rows, cols)

# Get lat/lon extent for axis labels
bounds = config["raster_info"]["bounds"]
crs = config["raster_info"]["crs"]
lon_min, lat_min, lon_max, lat_max = transform_bounds(
    crs, "EPSG:4326", bounds.left, bounds.bottom, bounds.right, bounds.top
)
extent_latlon = [lon_min, lon_max, lat_min, lat_max]

# Side-by-side susceptibility maps
cmap_prob = plt.cm.RdYlGn_r.copy()
cmap_prob.set_bad(color="lightgrey")

fig, axes = plt.subplots(1, 3, figsize=(20, 7), dpi=300)
maps = [prob_xgb_map, prob_rf_map, prob_svm_map]
titles = ["(a) XGBoost", "(b) Random Forest", "(c) SVM"]

for ax, data, title in zip(axes, maps, titles):
    im = ax.imshow(
        data, cmap=cmap_prob, vmin=0, vmax=1,
        interpolation="none", extent=extent_latlon, origin="upper",
    )
    ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax.set_xlabel("Longitude (\u00b0E)", fontsize=12)
    ax.set_ylabel("Latitude (\u00b0N)", fontsize=12)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f\u00b0"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f\u00b0"))

fig.subplots_adjust(right=0.92)
cbar = fig.colorbar(im, ax=axes, orientation="vertical",
                     fraction=0.03, pad=0.08, shrink=0.75)
cbar.set_label("Hotspot Susceptibility Probability", fontsize=12, labelpad=10)

fig.suptitle("Mapping of Hotspot Susceptibility \u2013 Ouagadougou",
             fontsize=16, fontweight="bold", y=0.96)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.savefig(FIG_DIR / "susceptibility_maps.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'susceptibility_maps.png'}")

In [ ]:
# Binary hotspot map (XGBoost)
nan_mask = np.isnan(data_3d[0])
pred_binary_masked = pred_map.copy()
pred_binary_masked[nan_mask] = np.nan

cmap_binary = mcolors.ListedColormap(["#27ae60", "#c0392b"])
norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5], cmap_binary.N)
cmap_binary.set_bad(color="#d5d8dc")

fig, ax = plt.subplots(figsize=(9, 8), dpi=300)
ax.imshow(pred_binary_masked, cmap=cmap_binary, norm=norm, interpolation="none")
ax.set_title("XGBoost \u2013 Binary Hotspot Classification\nOuagadougou",
             fontsize=14, fontweight="bold", pad=12)
ax.axis("off")

legend_elements = [
    Patch(facecolor="#27ae60", edgecolor="grey", label="Non-hotspot"),
    Patch(facecolor="#c0392b", edgecolor="grey", label="Hotspot"),
    Patch(facecolor="#d5d8dc", edgecolor="grey", label="No data"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=11,
          framealpha=0.9, edgecolor="grey")

plt.tight_layout()
plt.savefig(FIG_DIR / "binary_hotspot_map.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'binary_hotspot_map.png'}")